# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Diovalda22/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract — Content Opportunity Scoring

1. **Unit of Analysis (Grain):**  
   One row in the daily fact table = one `report_date` × `client_hash_id` × `content_hash_id`.  
   After aggregation for modeling: one row = one content item per client, aggregated over a 60-day observation window.

2. **Tables Used:**  
   - `fact_content_daily_performance` (partitioned by `month`, using `month=2026-02` + `month=2026-03` for development — two months to support 30-day feature and 30-day outcome windows)  
   - `dim_content` (content metadata: `word_count`, `content_type`)  
   - `dim_clients` (history coverage: `gsc_data_start`, `ga4_data_start`)

3. **Time Window:**  
   - **Feature window (prev30):** 2026-02-01 to 2026-02-28 (the preceding month).  
   - **Outcome window (last30):** 2026-03-02 to 2026-03-31 (the most recent 30 days of March).  
   - Iterating on `month=2026-02` + `month=2026-03` (mid-panel). The final month (June 2026) is sealed for testing.

4. **Prediction Target (Label / Proxy):**  
   - **Binary label:** `is_declining` — 1 if `imp_last30 < 0.8 × imp_prev30` (impressions dropped >20%), else 0.  
   - **Opportunity score (proxy):** `log1p(imp_prev30) × P(decline)` — ranks pages by both risk and visibility.

5. **Deliberately Excluded:**  
   - `imp_last30` as a feature (it IS the outcome window — using it would be target leakage).  
   - GA4 columns where `ga4_data_available IS NOT TRUE` (zeros there mean "no data", not "no engagement").  
   - Raw hash IDs (context only, never features).

In [5]:
import os, sys, getpass

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install duckdb huggingface_hub

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from dotenv import load_dotenv
        load_dotenv(os.path.join(os.path.dirname(os.path.abspath('.')), '..', '.env'))
        HF_TOKEN = os.environ.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb, numpy as np, pandas as pd

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-0[23]/*.parquet')",
}

print('Connected.')

Connected.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification

| Bucket | Fields | Notes |
|---|---|---|
| **Feature** | `imp_prev30`, `clk_prev30`, `ctr_prev30`, `pos_prev30`, `word_count` | All knowable before the outcome window |
| **Label** | `is_declining` (binary), `opportunity_score` (ranking proxy) | Computed from the outcome window |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `month` | For joins, splits, grouping — never model features |
| **Excluded** | `imp_last30` (it IS the label), GA4 columns where flag is not TRUE (zeros = missing, not zero engagement), raw IDs | Each encodes future or misleading information |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain check

Claim: one row = one `report_date × client_hash_id × content_hash_id`.  
Test: group by those three columns and look for duplicates. Zero rows back = grain holds.

In [6]:
grain_dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f'Duplicate rows found: {len(grain_dupes)}')
if len(grain_dupes) == 0:
    print('Grain holds: one row = one report_date × client × content.')
else:
    print(grain_dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found: 0
Grain holds: one row = one report_date × client × content.


### Query 2 — Row count and date span

Claim: month=2026-02 + month=2026-03 partitions cover Feb–March 2026.

In [ ]:
counts = con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content
    FROM {TABLES['fact_daily']}
""").df()

print(counts.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows   min_date   max_date  n_clients  n_content
   17196486 2026-02-01 2026-03-31         59     349411


### Query 3 — Availability check (IS TRUE filter)

Claim: `ga4_data_available` and `gsc_data_available` are three-valued (TRUE / FALSE / NULL).  
Filter with `IS TRUE` to avoid silent inclusion of zero-filled rows.

In [ ]:
avail = con.sql(f"""
    SELECT
        COUNT(*)                                                                    AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)                 AS gsc_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)                 AS ga4_available,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE
            THEN 1 ELSE 0 END)                                                      AS both_available,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                                        AS pct_gsc,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                                        AS pct_ga4
    FROM {TABLES['fact_daily']}
""").df()

print(avail.to_string(index=False))
print()
print('Only ~37% of rows have GSC data available (IS TRUE).')
print('Only ~4% have GA4 data available (IS TRUE).')
print('For Content Opportunity Scoring we filter to gsc_data_available IS TRUE.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available  ga4_available  both_available  pct_gsc  pct_ga4
   17196486      6232844.0       559287.0        489027.0     36.2      3.3

Only ~37% of rows have GSC data available (IS TRUE).
Only ~4% have GA4 data available (IS TRUE).
For Content Opportunity Scoring we filter to gsc_data_available IS TRUE.


### Build the five-feature frame + leakage trap

**Five features** (all from the prev-30 window — knowable before prediction):

| # | Feature | Available when? |
|---|---|---|
| 1 | `imp_prev30` — total GSC impressions, days 31–60 back | Knowable at decision moment: aggregated from the baseline window before the outcome period |
| 2 | `clk_prev30` — total GSC clicks, days 31–60 back | Same baseline window as above |
| 3 | `ctr_prev30` — click-through rate in baseline window | Derived from `clk_prev30 / imp_prev30`, both from baseline |
| 4 | `pos_prev30` — average GSC position in baseline window | Aggregated from daily position in the baseline window |
| 5 | `word_count` — article length from `dim_content` | Static metadata, known at content creation time |

In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    agg AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_clicks      ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_avg_position END)       AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_prev30 >= 10
    )
    SELECT
        a.*,
        COALESCE(d.word_count, 0) AS word_count
    FROM agg a
    LEFT JOIN {TABLES['dim_content']} d
        ON a.client_hash_id  = d.client_hash_id
       AND a.content_hash_id = d.content_hash_id
""").df()

features['ctr_prev30'] = np.where(
    features['imp_prev30'] > 0,
    features['clk_prev30'] / features['imp_prev30'] * 100,
    0.0
)

features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)

print(f'Feature frame: {len(features):,} content items')
print(f'Declining rate: {features["is_declining"].mean():.3f}')
features[['client_hash_id', 'content_hash_id',
           'imp_prev30', 'clk_prev30', 'ctr_prev30', 'pos_prev30', 'word_count',
           'is_declining']].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 120,507 content items
Declining rate: 0.281


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,ctr_prev30,pos_prev30,word_count,is_declining
0,client_62f4a7e64f5e0096,content_da0edf7bdf576b1a,2373.0,15.0,0.632111,4.415923,2571,0
1,client_62f4a7e64f5e0096,content_6e523f7e6429f722,1323.0,0.0,0.000000,6.814942,3957,1
2,client_62f4a7e64f5e0096,content_53d28019cceeccb5,293.0,3.0,1.023891,7.218437,4564,0
3,client_62f4a7e64f5e0096,content_c3c405058664da7c,1737.0,15.0,0.863558,3.225595,4217,0
4,client_62f4a7e64f5e0096,content_d8d29ffddb01d555,1297.0,2.0,0.154202,1.470705,3630,0
5,client_8dbf3abdf07569e0,content_121e3f7e8a310107,36.0,0.0,0.000000,8.500000,1458,1
6,client_9958f0a7ae1df715,content_b14b1b3afddc0287,106.0,0.0,0.000000,67.265152,0,0
7,client_9958f0a7ae1df715,content_94798aa612b0e422,893.0,1.0,0.111982,9.996110,2619,0
8,client_9958f0a7ae1df715,content_41b7a92fa1621773,979.0,1.0,0.102145,7.072467,2443,0
9,client_9958f0a7ae1df715,content_d0216c8ab6f6d0a4,110.0,0.0,0.000000,17.096154,0,0


### The Trap — Deliberate leakage experiment

We add `imp_last30` as a feature on purpose. This column is the outcome window — it is used
to compute the label. The model should score near-perfectly, proving it learned the label formula
rather than any real predictive signal. Then we remove it and keep the honest score.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['imp_prev30', 'clk_prev30', 'ctr_prev30', 'pos_prev30', 'word_count']
leaked_features = honest_features + ['imp_last30']

model_df = features.dropna(subset=honest_features).copy()
X = model_df[leaked_features]
y = model_df['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf_leaked = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
clf_leaked.fit(X_tr[leaked_features], y_tr)
auc_leaked = roc_auc_score(y_te, clf_leaked.predict_proba(X_te[leaked_features])[:, 1])

clf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
clf_honest.fit(X_tr[honest_features], y_tr)
auc_honest = roc_auc_score(y_te, clf_honest.predict_proba(X_te[honest_features])[:, 1])

print(f'AUC with leakage (imp_last30 included): {auc_leaked:.4f}')
print(f'AUC honest (imp_last30 removed):        {auc_honest:.4f}')
print()
if auc_leaked > auc_honest + 0.05:
    print('Leakage confirmed: imp_last30 inflates the score because it IS the outcome.')
    print('Keeping only the honest model.')
else:
    print('Difference is small — check label definition and feature windows.')

AUC with leakage (imp_last30 included): 0.9178
AUC honest (imp_last30 removed):        0.6713

Leakage confirmed: imp_last30 inflates the score because it IS the outcome.
Keeping only the honest model.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Limitation

**Unbalanced panel depth.** Not all 55 clients in March 2026 have the same history length.
Some clients joined the panel recently and have only a few weeks of data, while others have
17 months. This means:

- A global 60-day window may exceed a short-history client's available data, producing
  zero-filled prev30 aggregates that look like "no impressions" when the truth is "not tracked yet."
- Only ~37% of rows in this month have `gsc_data_available IS TRUE`, and only ~4% have
  `ga4_data_available IS TRUE`. GA4 engagement features are unavailable for most content items.
- The label (`is_declining`) is defined on GSC impressions only — it cannot capture engagement
  quality decline (e.g. a page still gets impressions but users bounce immediately).
- Conclusions from this month may not generalize to months with different seasonal patterns
  or client composition.

In [ ]:
print(f'Content items after GSC filter + imp_prev30 >= 10: {len(features):,}')
print(f'Clients represented: {features["client_hash_id"].nunique()}')
print(f'Declining rate: {features["is_declining"].mean():.3f}')
print(f'Honest AUC (5 features, no leakage): {auc_honest:.4f}')

Content items after GSC filter + imp_prev30 >= 10: 120,507
Clients represented: 42
Declining rate: 0.281
Honest AUC (5 features, no leakage): 0.6713


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.